In [ ]:
import warnings
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from metric import precision_at_recall

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

Загружаем данные

In [ ]:
train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print(f"Размер Train:  {train.shape}")
print(f"Размер Test:   {test.shape}")
print(f"Размер Events: {events.shape}")
print(f"Доля ботов в Train: {train['target'].mean():.4f}")

Предобработка и очистка данных

In [ ]:
#Удаление повторных отправлений кликов (дубликатов)
initial_len = len(events)
events = events.drop_duplicates(subset=['cookie_id', 'event_ts', 'eid', 'item_id'])
print(f"Удалено дубликатов событий: {initial_len - len(events)}")

#Расчет мета-признаков возраста куки на момент окна
train['cookie_age_sec'] = (train['window_start_ts'] - train['cookie_created_at']).dt.total_seconds()
test['cookie_age_sec'] = (test['window_start_ts'] - test['cookie_created_at']).dt.total_seconds()

meta_all = pd.concat([
    train[['cookie_id', 'window_start_ts', 'window_end_ts']], 
    test[['cookie_id', 'window_start_ts', 'window_end_ts']]
], axis=0, ignore_index=True)

#Мердж и фильтрация событий строго внутри окна наблюдения
ev_merged = events.merge(meta_all, on='cookie_id', how='inner')
ev = ev_merged[
    (ev_merged['event_ts'] >= ev_merged['window_start_ts']) & 
    (ev_merged['event_ts'] < ev_merged['window_end_ts'])
].copy()

del ev_merged, events

#Хронологическая сортировка событий
ev = ev.sort_values(['cookie_id', 'event_ts']).reset_index(drop=True)
print(f"Осталось очищенных событий в окне: {len(ev)}")

Future Engeenring

In [ ]:
#Временные дельты между последовательными событиями
ev['dt'] = ev.groupby('cookie_id')['event_ts'].diff().dt.total_seconds()

#Динамика и скорость движения курсора
ev['dx'] = ev.groupby('cookie_id')['pointer_x'].diff()
ev['dy'] = ev.groupby('cookie_id')['pointer_y'].diff()
ev['dist'] = np.sqrt(ev['dx']**2 + ev['dy']**2)
ev['speed'] = ev['dist'] / (ev['dt'] + 1e-5)

#Матрица типов событий
eid_raw = pd.crosstab(ev['cookie_id'], ev['eid'])
eid_counts = eid_raw.add_prefix('eid_cnt_').reset_index()
eid_ratios = eid_raw.div(eid_raw.sum(axis=1), axis=0).add_prefix('eid_ratio_').reset_index()

#Временные агрегаты и статистика курсора
time_aggs = ev.groupby('cookie_id').agg(
    dt_mean=('dt', 'mean'),
    dt_std=('dt', 'std'),
    dt_min=('dt', 'min'),
    dt_max=('dt', 'max'),
    dt_median=('dt', 'median'),
    dt_skew=('dt', 'skew'),
    dt_zero_cnt=('dt', lambda x: (x == 0).sum()),
    dt_fast_cnt=('dt', lambda x: (x < 0.15).sum()),
    dist_mean=('dist', 'mean'),
    dist_max=('dist', 'max'),
    speed_max=('speed', 'max'),
    speed_mean=('speed', 'mean'),
    has_pointer_cnt=('pointer_x', 'count')
).reset_index()

#Коэффициент вариации времени (CV = std / mean) - главный индикатор роботов
time_aggs['dt_cv'] = time_aggs['dt_std'] / (time_aggs['dt_mean'] + 1e-5)

#Основные структурные агрегаты
main_aggs = ev.groupby('cookie_id').agg(
    n_events=('event_ts', 'count'),
    n_unique_items=('item_id', 'nunique'),
    n_unique_cats=('item_category', 'nunique'),
    n_unique_locs=('item_location', 'nunique'),
    n_unique_queries=('search_query', 'nunique'),
    n_unique_pages=('search_page', 'nunique'),
    max_search_page=('search_page', 'max'),
    first_event=('event_ts', 'min'),
    last_event=('event_ts', 'max')
).reset_index()

#Парсинг строки User-Agent
ua_df = ev[['cookie_id', 'user_agent']].drop_duplicates(subset=['cookie_id'])
ua_str = ua_df['user_agent'].fillna('').astype(str).str.lower()
ua_df['ua_is_null'] = ua_df['user_agent'].isna().astype(int)
ua_df['ua_len'] = ua_str.str.len()
ua_df['ua_bot_kw'] = ua_str.str.contains(
    'python|scrapy|requests|headless|selenium|bot|crawler|puppeteer|curl|go-http|axios|urllib', 
    regex=True
).astype(int)
ua_df = ua_df.drop(columns=['user_agent'])

print("Признаки успешно извлечены!")

Сборка признаков и учет кук без активности

In [ ]:
def assemble_dataset(df_meta):
    df = df_meta.copy()
    
    #Мерджим созданные группы признаков
    df = df.merge(main_aggs, on='cookie_id', how='left')
    df = df.merge(time_aggs, on='cookie_id', how='left')
    df = df.merge(eid_counts, on='cookie_id', how='left')
    df = df.merge(eid_ratios, on='cookie_id', how='left')
    df = df.merge(ua_df, on='cookie_id', how='left')
    
    #Задержки сессии
    df['delay_start_sec'] = (df['first_event'] - df['window_start_ts']).dt.total_seconds()
    df['active_duration_sec'] = (df['last_event'] - df['first_event']).dt.total_seconds()
    
    #Относительные коэффициенты
    df['item_per_event'] = df['n_unique_items'] / (df['n_events'] + 1e-5)
    df['cat_per_item'] = df['n_unique_cats'] / (df['n_unique_items'] + 1e-5)
    df['pointer_ratio'] = df['has_pointer_cnt'] / (df['n_events'] + 1e-5)
    df['zero_dt_ratio'] = df['dt_zero_cnt'] / (df['n_events'] + 1e-5)
    df['fast_dt_ratio'] = df['dt_fast_cnt'] / (df['n_events'] + 1e-5)
    
    #Флаг присутствия активности
    df['has_events'] = df['n_events'].notna().astype(int)
    
    drop_cols = ['cookie_id', 'cookie_created_at', 'window_start_ts', 'window_end_ts', 
                 'first_event', 'last_event', 'target']
    drop_cols = [c for c in drop_cols if c in df.columns]
    
    return df.drop(columns=drop_cols)

#Сборка финальных выборок
X_train_df = assemble_dataset(train)
X_test_df = assemble_dataset(test)
y_train_full = train['target'].values

feature_cols = [c for c in X_train_df.columns if c in X_test_df.columns]
X_train_full = X_train_df[feature_cols].fillna(-999)
X_test_full = X_test_df[feature_cols].fillna(-999)

print(f"Итого чистовых признаков в модели: {len(feature_cols)}")

Валидация по времени

In [ ]:
is_valid = train['window_start_ts'].ge('2026-04-17').values

X_tr, y_tr = X_train_full.loc[~is_valid], y_train_full[~is_valid]
X_va, y_va = X_train_full.loc[is_valid], y_train_full[is_valid]

# Обучаем 3 алгоритма бустинга
lgb = LGBMClassifier(n_estimators=1200, learning_rate=0.02, num_leaves=63, max_depth=8, subsample=0.8, colsample_bytree=0.7, random_state=SEED, verbose=-1)
cat = CatBoostClassifier(iterations=1200, learning_rate=0.03, depth=7, random_seed=SEED, verbose=0)
xgb = XGBClassifier(n_estimators=1000, learning_rate=0.02, max_depth=6, random_state=SEED, eval_metric='logloss')

lgb.fit(X_tr, y_tr)
cat.fit(X_tr, y_tr)
xgb.fit(X_tr, y_tr)

# Ранговое усреднение переводим вероятности в процентили
r_lgb_v = pd.Series(lgb.predict_proba(X_va)[:, 1]).rank(pct=True).values
r_cat_v = pd.Series(cat.predict_proba(X_va)[:, 1]).rank(pct=True).values
r_xgb_v = pd.Series(xgb.predict_proba(X_va)[:, 1]).rank(pct=True).values

# Блендинг рангов + случайный микрошум для разрыва одинаковых скоров 
p_val_raw = (r_lgb_v + r_cat_v + r_xgb_v) / 3.0
p_val_raw += np.random.uniform(0, 1e-10, size=len(p_val_raw))
p_val = (p_val_raw - p_val_raw.min()) / (p_val_raw.max() - p_val_raw.min() + 1e-12)

val_score = precision_at_recall(y_va, p_val)

print(f"Валидационный результат P@R>=0.7: {val_score:.5f}")

Обучение финального ансамбля и генерация submission.csv

In [ ]:
lgb_full = LGBMClassifier(n_estimators=1200, learning_rate=0.02, num_leaves=63, max_depth=8, subsample=0.8, colsample_bytree=0.7, random_state=SEED, verbose=-1)
cat_full = CatBoostClassifier(iterations=1200, learning_rate=0.03, depth=7, random_seed=SEED, verbose=0)
xgb_full = XGBClassifier(n_estimators=1000, learning_rate=0.02, max_depth=6, random_state=SEED, eval_metric='logloss')

lgb_full.fit(X_train_full, y_train_full)
cat_full.fit(X_train_full, y_train_full)
xgb_full.fit(X_train_full, y_train_full)

#Получаем ранги на тесте
r_lgb_t = pd.Series(lgb_full.predict_proba(X_test_full)[:, 1]).rank(pct=True).values
r_cat_t = pd.Series(cat_full.predict_proba(X_test_full)[:, 1]).rank(pct=True).values
r_xgb_t = pd.Series(xgb_full.predict_proba(X_test_full)[:, 1]).rank(pct=True).values

p_test_raw = (r_lgb_t + r_cat_t + r_xgb_t) / 3.0
p_test_raw += np.random.uniform(0, 1e-10, size=len(p_test_raw))

#Нормализация скоров строго от 0.0 до 1.0
final_scores = (p_test_raw - p_test_raw.min()) / (p_test_raw.max() - p_test_raw.min() + 1e-12)
final_scores = np.clip(final_scores, 0.0, 1.0)

#Формирование сабмита
sub = pd.DataFrame({
    'cookie_id': test['cookie_id'],
    'score': final_scores
})

#проверка корректности
assert len(sub) == len(test), "Ошибка: Несовпадение количества строк"
assert sub['score'].between(0.0, 1.0).all(), "Ошибка: Scores выходят за пределы [0, 1]!"
assert not sub['score'].isna().any(), "Ошибка: Найдены пропущенные значения"

sub.to_csv('submission.csv', index=False)

1. Подготовка данных
Убрал дубликаты: Изучая логи, заметил, что браузер иногда отправляет один и тот же клик дважды из-за сбоев сети. Удалил 5 000+ таких ложных дублей, чтобы не путать модель
Отрезал лишнее время: Смотрел только на события, которые произошли строго внутри суточного окна. Всё, что до или после отбросил
Учел куки-пустышки: дал модели сигнал: has_events = 0 (то есть активности не было)

2. Создание фичей
Создал 51 признак. Вот 4 самых главных:
Разброс пауз между кликами (dt_cv):
У бота: задержка закодирована как sleep(1), поэтому все паузы одинаковые
У человека: задержки прыгают то 0.3 секунды, то 5 секунд
Структура действий (Матрица EID):
Модель смотрит на процент каждого действия. Если у пользователя 100% действий то это просто просмотр страницы и 0% просмотров телефона/чата, то это почти наверняка бот-парсер
Движения мышки:
У ботов либо вообще нет координат мыши, либо курсор мгновенно прыгает из точки А в точку Б с космической скоростью
User-Agent:
Проверили текст браузера на подозрительные слова: python, selenium, headless, requests, puppeteer

3. Валидация
Отделилл последние дни обучающей выборки (после 17 апреля) и использовал их для проверки. Модель обучалась на прошлом и тестировалась на будущем
Результат: локальная проверка показывала 0.67, а на скрытом лидерборде результат вырос до 0.6976. Это значит, что модель не переобучилась и отлично работает на новых данных

4. Финальная модель
3 лучших алгоритма: LightGBM, CatBoost и XGBoost
Перевел предсказания каждой модели в места в рейтинге (от 1-го до последнего). И сложил именно места! Это дало идеальный порядок кук от самого подозрительного до самого похожего на человека

5. Сравнение решений
Baseline: Смотрела только на общее число кликов
Результат: 0.15
Простая модель (LightGBM без чистки логов): Считала базовые агрегаты, путалась в одинаковых скорах
Результат: 0.655
Финальное решение (3 модели + Ранги + 51 фича):
Результат: 0.69762 на лидерборде
